In [3]:
# Cell 1: imports & global config

import os, math, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader, Dataset

import matplotlib.pyplot as plt
import pandas as pd

# 统一随机性，便于对比
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# FAST-DEV 开关：True=小实验；False=较大规模
FAST_DEV = True

# 统一管理实验规模
if FAST_DEV:
    DATA_FRACTION = 0.05        # 用 5% 数据
    MAX_ITERS     = 1500        # 如果后面要正式训练可用
    EVAL_INTERVAL = 100
    BATCH_SIZE    = 32
    N_LAYERS      = 2
    N_HEADS       = 2
    D_MODEL       = 128
    D_FF          = 4 * D_MODEL
    CONTEXT_LEN   = 64          # 默认 L
    LR            = 3e-4
    WARMUP_STEPS  = 200
else:
    DATA_FRACTION = 1.0
    MAX_ITERS     = 100_000
    EVAL_INTERVAL = 1000
    BATCH_SIZE    = 64
    N_LAYERS      = 6
    N_HEADS       = 8
    D_MODEL       = 512
    D_FF          = 4 * D_MODEL
    CONTEXT_LEN   = 128
    LR            = 3e-4
    WARMUP_STEPS  = 2000


Device: cpu


In [4]:
# Cell 2: load text8 data (train/test)

train_path = "./data/text8_train.txt"
test_path  = "./data/text8_test.txt"

with open(train_path, "r") as f:
    train_text = f.read()
with open(test_path, "r") as f:
    test_text = f.read()

print(f"Original lengths: train={len(train_text):_}, test={len(test_text):_}")

if DATA_FRACTION < 1.0:
    train_text = train_text[:int(len(train_text) * DATA_FRACTION)]
    test_text  = test_text[:int(len(test_text) * DATA_FRACTION)]
    print(f"Using subset: train={len(train_text):_}, test={len(test_text):_}")

print("Sample training text preview:", repr(train_text[:200]))


Original lengths: train=90_000_000, test=5_000_000
Using subset: train=4_500_000, test=250_000
Sample training text preview: ' anarchism originated as a term of abuse first used against early working class radicals including the diggers of the english revolution and the sans culottes of the french revolution whilst the term '


In [5]:
# Cell 3: tokenizer & encode/decode & splits

# 1) 字符表
chars = sorted(list(set(train_text + test_text)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(chars)

print(f"vocab_size = {vocab_size}")
print("vocab preview:", chars[:50])

# 2) 编码/解码函数
def encode(s: str) -> torch.Tensor:
    return torch.tensor([stoi[c] for c in s], dtype=torch.long)

def decode(ids) -> str:
    return "".join(itos[int(i)] for i in ids)

# 3) 文本 -> ids
train_ids = encode(train_text)
test_ids  = encode(test_text)

# 4) 从训练集切出一个小验证集
val_ratio = 0.1
split = int(len(train_ids) * (1 - val_ratio))
train_data = train_ids[:split]
val_data   = train_ids[split:]

print(f"train_data={len(train_data):_}, val_data={len(val_data):_}, test_data={len(test_ids):_}")


vocab_size = 27
vocab preview: [' ', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
train_data=4_050_000, val_data=450_000, test_data=250_000


In [6]:
# Cell 4: batch sampling function

def get_batch(split, batch_size=BATCH_SIZE, block_size=CONTEXT_LEN):
    """
    从 train_data 或 val_data 中随机抽取 batch_size 个长度为 block_size 的片段。
    x: (B, block_size)
    y: (B, block_size)，为下一字符标签。
    """
    data_ = train_data if split == "train" else val_data
    ix = torch.randint(0, len(data_) - block_size - 1, (batch_size,))
    x = torch.stack([data_[i:i+block_size]     for i in ix])
    y = torch.stack([data_[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)


In [7]:
# Cell 5: baseline TinyGPT (causal transformer)

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, block_size, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
        # 下三角 mask
        self.register_buffer(
            "mask",
            torch.tril(torch.ones(block_size, block_size))
                .view(1, 1, block_size, block_size)
        )

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]  # (B, nH, T, Hd)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_drop(att)
        y = att @ v  # (B, nH, T, Hd)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_drop(self.proj(y))
        return y

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, block_size, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, block_size, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class TinyGPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_layers, n_heads, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, block_size, dropout)
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.block_size = block_size

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device).unsqueeze(0)
        x = self.token_emb(idx) + self.pos_emb(pos)
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        logits = self.head(x)  # (B, T, vocab)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1)
            )
        return logits, loss


In [10]:
# Cell 6: import variant models & define build_model / forward_for_training

# 运行你写好的 PyTorch variants notebook
%run ./models/models_pytorch_variants.ipynb

# 注意：此处不再需要 from models_pytorch_variants import ...
# 因为上面 %run 之后，下面这些类已经在当前命名空间里可以直接用了：
# DecoderOnlyTransformerMQPT, XLMemoryTransformerPT, MoETransformerPT, RelPosLocalGlobalTransformerPT

def build_model(
    variant: str,
    vocab_size: int,
    block_size: int,
    n_layers: int,
    n_heads: int,
    d_model: int,
    d_ff: int,
    dropout: float = 0.1,
):
    if variant == "tinygpt":
        return TinyGPT(
            vocab_size=vocab_size,
            block_size=block_size,
            n_layers=n_layers,
            n_heads=n_heads,
            d_model=d_model,
            d_ff=d_ff,
            dropout=dropout,
        )

    elif variant == "mq_rope":
        return DecoderOnlyTransformerMQPT(
            vocab_size=vocab_size,
            d_model=d_model,
            n_layers=n_layers,
            n_heads=n_heads,
            n_kv_heads=1,
            max_len=block_size,
            dropout=dropout,
        )

    elif variant == "xl_memory":
        return XLMemoryTransformerPT(
            vocab_size=vocab_size,
            d_model=d_model,
            n_layers=n_layers,
            n_heads=n_heads,
            mem_len=block_size,
            dropout=dropout,
            mlp_mult=d_ff / d_model if d_model > 0 else 2.667,
        )

    elif variant == "moe":
        return MoETransformerPT(
            vocab_size=vocab_size,
            d_model=d_model,
            n_layers=n_layers,
            n_heads=n_heads,
            n_experts=4,
            dropout=dropout,
            moe_mult=d_ff / d_model if d_model > 0 else 2.667,
            max_len=block_size,
        )

    elif variant == "local_global":
        return RelPosLocalGlobalTransformerPT(
            vocab_size=vocab_size,
            d_model=d_model,
            n_layers=n_layers,
            n_heads=n_heads,
            window=block_size,
            n_global=8,
            dropout=dropout,
        )

    else:
        raise ValueError(f"Unknown model variant: {variant}")


def forward_for_training(model, x, y):
    if isinstance(model, XLMemoryTransformerPT):
        logits, loss, _ = model(x, mems=None, targets=y)
        return logits, loss

    if isinstance(model, MoETransformerPT):
        logits, loss, _ = model(x, targets=y)
        return logits, loss

    logits, loss = model(x, y)
    return logits, loss


In [11]:
# Cell 7: hyperparameter search space

# 搜索范围（你可以根据实际情况缩小/调整）
SEARCH = {
    "LR": [1e-3, 5e-4, 3e-4, 1e-4],           # 学习率
    "WARMUP_STEPS": [100, 300, 1000],         # warmup 步数
    "CONTEXT_LEN": [32, 64, 128],             # 上下文窗口长度 L
}

DEFAULTS = {
    "LR": LR,
    "WARMUP_STEPS": WARMUP_STEPS,
    "BATCH_SIZE": BATCH_SIZE,
    "CONTEXT_LEN": CONTEXT_LEN,
    "N_LAYERS": N_LAYERS,
    "N_HEADS": N_HEADS,
    "D_MODEL": D_MODEL,
    "D_FF": D_FF,
}

grid = []
for lr in SEARCH["LR"]:
    for warm in SEARCH["WARMUP_STEPS"]:
        for L in SEARCH["CONTEXT_LEN"]:
            cfg = DEFAULTS.copy()
            cfg.update({"LR": lr, "WARMUP_STEPS": warm, "CONTEXT_LEN": L})
            grid.append(cfg)

print(f"共 {len(grid)} 组实验配置")


共 36 组实验配置


In [12]:
# Cell 8: train_and_eval for a given variant + helper to run grid

def train_and_eval(cfg, steps=800, seed=42, variant="tinygpt"):
    """
    用给定 cfg + 指定 variant 训练 steps 步，然后返回:
      - val_acc_last
      - time_sec
    """
    set_seed(seed)

    model_ = build_model(
        variant=variant,
        vocab_size=vocab_size,
        block_size=cfg["CONTEXT_LEN"],
        n_layers=cfg["N_LAYERS"],
        n_heads=cfg["N_HEADS"],
        d_model=cfg["D_MODEL"],
        d_ff=cfg["D_FF"],
        dropout=0.1,
    ).to(device)

    optimizer = torch.optim.AdamW(
        model_.parameters(),
        lr=cfg["LR"],
        betas=(0.9, 0.95),
        weight_decay=0.1,
    )

    def lr_schedule(it):
        if it < cfg["WARMUP_STEPS"]:
            return cfg["LR"] * (it + 1) / max(1, cfg["WARMUP_STEPS"])
        progress = (it - cfg["WARMUP_STEPS"]) / max(1, steps - cfg["WARMUP_STEPS"])
        return 0.1 * cfg["LR"] + 0.9 * cfg["LR"] * 0.5 * (1 + math.cos(math.pi * progress))

    model_.train()
    t0 = time.time()
    for it in range(steps):
        for pg in optimizer.param_groups:
            pg["lr"] = lr_schedule(it)

        xb, yb = get_batch("train", batch_size=cfg["BATCH_SIZE"], block_size=cfg["CONTEXT_LEN"])
        _, loss = forward_for_training(model_, xb, yb)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_.parameters(), 1.0)
        optimizer.step()

    elapsed = time.time() - t0

    @torch.no_grad()
    def val_acc_last():
        model_.eval()
        x, y = get_batch("val", batch_size=256, block_size=cfg["CONTEXT_LEN"])
        logits, _ = forward_for_training(model_, x, y)
        preds = logits.argmax(dim=-1)
        return (preds[:, -1] == y[:, -1]).float().mean().item()

    acc = val_acc_last()
    return {"val_acc_last": acc, "time_sec": elapsed}


def run_grid_for_variant(variant, steps=800, seed=42):
    """
    对某个模型 variant，在同一 grid 上跑一遍小网格搜索。
    返回一个 list[dict]，每个 dict 包含:
      - MODEL_VARIANT
      - LR / WARMUP_STEPS / CONTEXT_LEN / ...
      - val_acc_last / time_sec
    """
    results = []
    for i, cfg in enumerate(grid, 1):
        print(f"[{variant}] [{i}/{len(grid)}] LR={cfg['LR']}, Warmup={cfg['WARMUP_STEPS']}, L={cfg['CONTEXT_LEN']}")
        out = train_and_eval(cfg, steps=steps, seed=seed, variant=variant)
        row = {**cfg, "MODEL_VARIANT": variant, **out}
        print(f"  → acc_last={row['val_acc_last']*100:.2f}%, time={row['time_sec']:.1f}s")
        results.append(row)
    return results


In [13]:
# Cell 9: run grid for multiple variants and save combined results

VARIANTS = [
    "tinygpt",
    "mq_rope",
    "moe",
    "local_global",
    # "xl_memory",  # 如果想连 memory 版也调，可以再加上它
]

all_results = []
for v in VARIANTS:
    res_v = run_grid_for_variant(v, steps=800, seed=42)
    all_results.extend(res_v)

df_all = (
    pd.DataFrame(all_results)
      .sort_values(["MODEL_VARIANT", "val_acc_last"], ascending=[True, False])
      .reset_index(drop=True)
)

display(df_all.head(20))

csv_path = "mini_grid_results_all_models.csv"
df_all.to_csv(csv_path, index=False)
print("所有模型的 mini-grid 结果已保存到:", csv_path)


[tinygpt] [1/36] LR=0.001, Warmup=100, L=32
  → acc_last=35.94%, time=13.0s
[tinygpt] [2/36] LR=0.001, Warmup=100, L=64
  → acc_last=33.59%, time=27.9s
[tinygpt] [3/36] LR=0.001, Warmup=100, L=128
  → acc_last=32.03%, time=74.7s
[tinygpt] [4/36] LR=0.001, Warmup=300, L=32
  → acc_last=36.33%, time=16.8s
[tinygpt] [5/36] LR=0.001, Warmup=300, L=64
  → acc_last=34.38%, time=33.5s
[tinygpt] [6/36] LR=0.001, Warmup=300, L=128
  → acc_last=31.64%, time=78.4s
[tinygpt] [7/36] LR=0.001, Warmup=1000, L=32
  → acc_last=34.38%, time=17.1s
[tinygpt] [8/36] LR=0.001, Warmup=1000, L=64
  → acc_last=35.16%, time=34.4s
[tinygpt] [9/36] LR=0.001, Warmup=1000, L=128
  → acc_last=25.00%, time=80.0s
[tinygpt] [10/36] LR=0.0005, Warmup=100, L=32
  → acc_last=32.03%, time=18.4s
[tinygpt] [11/36] LR=0.0005, Warmup=100, L=64
  → acc_last=28.52%, time=33.4s
[tinygpt] [12/36] LR=0.0005, Warmup=100, L=128
  → acc_last=26.17%, time=78.9s
[tinygpt] [13/36] LR=0.0005, Warmup=300, L=32
  → acc_last=34.77%, time=17.

,LR,WARMUP_STEPS,BATCH_SIZE,CONTEXT_LEN,N_LAYERS,N_HEADS,D_MODEL,D_FF,MODEL_VARIANT,val_acc_last,time_sec
0,0.0010,300,32,32,2,2,128,512,local_global,0.316406,19.782257
1,0.0010,100,32,32,2,2,128,512,local_global,0.312500,19.931132
2,0.0005,300,32,32,2,2,128,512,local_global,0.304688,19.742858
3,0.0003,1000,32,32,2,2,128,512,local_global,0.304688,24.114748
4,0.0005,100,32,32,2,2,128,512,local_global,0.300781,20.276380
5,0.0003,100,32,32,2,2,128,512,local_global,0.300781,24.256654
6,0.0003,300,32,32,2,2,128,512,local_global,0.300781,23.603677
7,0.0010,100,32,128,2,2,128,512,local_global,0.281250,82.645292
8,0.0005,100,32,128,2,2,128,512,local_global,0.281250,82.327169
9,0.0005,300,32,128,2,2,128,512,local_global,0.281250,81.249807


所有模型的 mini-grid 结果已保存到: mini_grid_results_all_models.csv


In [15]:
# Cell 10: Round 2 search for all variants (jitter LR/WARMUP and tune beta2)

import math, time, torch
import pandas as pd

# beta2 备选：包含原版常用 0.999
BETA2_CANDIDATES = [0.90, 0.95, 0.98, 0.999]

# ✅ 和第一轮保持一致：steps 用 800，而不是 600
STEPS2 = 800   # 每个组合跑多少步（和 round1 一样）

########################################
# 1. 从第一轮结果里，为每个模型选 base configs
########################################

# 读入第一轮的总结果（你之前保存的那个）
df1 = pd.read_csv("mini_grid_results_all_models.csv")

# 按模型内部从高到低排
df1 = df1.sort_values(["MODEL_VARIANT", "val_acc_last"], ascending=[True, False])

TOP_K = 2   # 每个模型取前 K 个作为 base configs，可以改成 1 或 3

base_rows = df1.groupby("MODEL_VARIANT").head(TOP_K).reset_index(drop=True)

print("Base configs (per model, top-K from round 1):")
display(base_rows[["MODEL_VARIANT", "LR", "WARMUP_STEPS", "CONTEXT_LEN", "val_acc_last"]])

base_configs = []
for _, row in base_rows.iterrows():
    base_configs.append({
        "MODEL_VARIANT": row["MODEL_VARIANT"],
        "LR":           float(row["LR"]),
        "WARMUP_STEPS": int(row["WARMUP_STEPS"]),
        "CONTEXT_LEN":  int(row["CONTEXT_LEN"]),
        # 下面这些一般在 df1 里也有，如果没有，就用全局默认
        "N_LAYERS":     int(row["N_LAYERS"]) if "N_LAYERS" in row else N_LAYERS,
        "N_HEADS":      int(row["N_HEADS"])  if "N_HEADS"  in row else N_HEADS,
        "D_MODEL":      int(row["D_MODEL"])  if "D_MODEL"  in row else D_MODEL,
        "D_FF":         int(row["D_FF"])     if "D_FF"     in row else D_FF,
        "BATCH_SIZE":   int(row["BATCH_SIZE"]) if "BATCH_SIZE" in row else BATCH_SIZE,
    })

print("\nCollected base_configs:")
for cfg in base_configs:
    print(cfg)

########################################
# 2. 构造第二轮搜索网格：在每个 base 配置周围抖一抖 LR / WARMUP + 加不同 beta2
########################################

second_grid = []

for base in base_configs:
    base_lr   = base["LR"]
    base_warm = base["WARMUP_STEPS"]
    L         = base["CONTEXT_LEN"]
    variant   = base["MODEL_VARIANT"]

    # 在 base_lr 周围抖一抖（包含 base_lr 本身）
    lr_candidates = sorted(set([
        base_lr,
        base_lr * 0.7,
        base_lr * 1.3,
    ]))

    # 在 base_warm 周围抖一抖（包含 base_warm 本身）
    warm_candidates = sorted(set([
        base_warm,
        max(50, base_warm // 2),
        base_warm * 2,
    ]))

    for lr in lr_candidates:
        for warm in warm_candidates:
            for beta2 in BETA2_CANDIDATES:
                cfg = {
                    "MODEL_VARIANT": variant,
                    "LR":           float(lr),
                    "WARMUP_STEPS": int(warm),
                    "CONTEXT_LEN":  int(L),
                    "BETA2":        float(beta2),
                    "BATCH_SIZE":   base["BATCH_SIZE"],
                    "N_LAYERS":     base["N_LAYERS"],
                    "N_HEADS":      base["N_HEADS"],
                    "D_MODEL":      base["D_MODEL"],
                    "D_FF":         base["D_FF"],
                }
                second_grid.append(cfg)

print(f"\nRound 2: total {len(second_grid)} configs to run.")

########################################
# 3. 定义第二轮的训练 + 验证函数（支持任意模型 variant）
########################################

def train_and_eval_round2(cfg, steps=STEPS2, seed=42):
    set_seed(seed)

    # 根据 MODEL_VARIANT 构建对应的模型
    model_ = build_model(
        variant   = cfg["MODEL_VARIANT"],
        vocab_size= vocab_size,
        block_size= cfg["CONTEXT_LEN"],
        n_layers  = cfg["N_LAYERS"],
        n_heads   = cfg["N_HEADS"],
        d_model   = cfg["D_MODEL"],
        d_ff      = cfg["D_FF"],
        dropout   = 0.1,
    ).to(device)

    # ✅ 和第一轮保持一致：weight_decay=0.1
    optimizer = torch.optim.AdamW(
        model_.parameters(),
        lr    = cfg["LR"],
        betas = (0.9, cfg["BETA2"]),
        weight_decay = 0.1,
    )

    def lr_schedule(it):
        if it < cfg["WARMUP_STEPS"]:
            return cfg["LR"] * (it + 1) / max(1, cfg["WARMUP_STEPS"])
        progress = (it - cfg["WARMUP_STEPS"]) / max(1, steps - cfg["WARMUP_STEPS"])
        return 0.1 * cfg["LR"] + 0.9 * cfg["LR"] * 0.5 * (1 + math.cos(math.pi * progress))

    model_.train()
    t0 = time.time()
    for it in range(steps):
        # 更新学习率
        lr_now = lr_schedule(it)
        for pg in optimizer.param_groups:
            pg["lr"] = lr_now

        xb, yb = get_batch("train",
                           batch_size=cfg["BATCH_SIZE"],
                           block_size=cfg["CONTEXT_LEN"])
        logits, loss = forward_for_training(model_, xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_.parameters(), 1.0)
        optimizer.step()

    elapsed = time.time() - t0

    @torch.no_grad()
    def val_acc_last():
        model_.eval()
        x, y = get_batch("val", batch_size=256, block_size=cfg["CONTEXT_LEN"])
        logits, _ = forward_for_training(model_, x, y)
        preds = logits.argmax(dim=-1)
        return (preds[:, -1] == y[:, -1]).float().mean().item()

    acc = val_acc_last()
    return {"val_acc_last": acc, "time_sec": elapsed}

########################################
# 4. 实际跑第二轮网格 & 保存结果
########################################

second_round_results = []
for i, cfg in enumerate(second_grid, 1):
    print(f"[Round 2: {i}/{len(second_grid)}] variant={cfg['MODEL_VARIANT']}, "
          f"LR={cfg['LR']}, warmup={cfg['WARMUP_STEPS']}, L={cfg['CONTEXT_LEN']}, beta2={cfg['BETA2']}")
    out = train_and_eval_round2(cfg)
    row = {**cfg, **out}
    print(f"  -> acc_last={row['val_acc_last']*100:.2f}%, time={row['time_sec']:.1f}s")
    second_round_results.append(row)

df2 = (
    pd.DataFrame(second_round_results)
      .sort_values(["MODEL_VARIANT", "val_acc_last"], ascending=[True, False])
      .reset_index(drop=True)
)

print("\n=== Round 2 results (per model, sorted by val_acc_last) ===")
display(df2.head(20))

df2.to_csv("second_round_results_all_models.csv", index=False)
print("Saved second-round results for all models to second_round_results_all_models.csv")


Base configs (per model, top-K from round 1):


,MODEL_VARIANT,LR,WARMUP_STEPS,CONTEXT_LEN,val_acc_last
0,local_global,0.0010,300,32,0.316406
1,local_global,0.0010,100,32,0.312500
2,moe,0.0010,300,64,0.296875
3,moe,0.0005,100,64,0.296875
4,mq_rope,0.0010,300,64,0.378906
5,mq_rope,0.0010,1000,64,0.375000
6,tinygpt,0.0010,300,32,0.363281
7,tinygpt,0.0010,100,32,0.359375



Collected base_configs:
{'MODEL_VARIANT': 'local_global', 'LR': 0.001, 'WARMUP_STEPS': 300, 'CONTEXT_LEN': 32, 'N_LAYERS': 2, 'N_HEADS': 2, 'D_MODEL': 128, 'D_FF': 512, 'BATCH_SIZE': 32}
{'MODEL_VARIANT': 'local_global', 'LR': 0.001, 'WARMUP_STEPS': 100, 'CONTEXT_LEN': 32, 'N_LAYERS': 2, 'N_HEADS': 2, 'D_MODEL': 128, 'D_FF': 512, 'BATCH_SIZE': 32}
{'MODEL_VARIANT': 'moe', 'LR': 0.001, 'WARMUP_STEPS': 300, 'CONTEXT_LEN': 64, 'N_LAYERS': 2, 'N_HEADS': 2, 'D_MODEL': 128, 'D_FF': 512, 'BATCH_SIZE': 32}
{'MODEL_VARIANT': 'moe', 'LR': 0.0005, 'WARMUP_STEPS': 100, 'CONTEXT_LEN': 64, 'N_LAYERS': 2, 'N_HEADS': 2, 'D_MODEL': 128, 'D_FF': 512, 'BATCH_SIZE': 32}
{'MODEL_VARIANT': 'mq_rope', 'LR': 0.001, 'WARMUP_STEPS': 300, 'CONTEXT_LEN': 64, 'N_LAYERS': 2, 'N_HEADS': 2, 'D_MODEL': 128, 'D_FF': 512, 'BATCH_SIZE': 32}
{'MODEL_VARIANT': 'mq_rope', 'LR': 0.001, 'WARMUP_STEPS': 1000, 'CONTEXT_LEN': 64, 'N_LAYERS': 2, 'N_HEADS': 2, 'D_MODEL': 128, 'D_FF': 512, 'BATCH_SIZE': 32}
{'MODEL_VARIANT': 'tiny

,MODEL_VARIANT,LR,WARMUP_STEPS,CONTEXT_LEN,BETA2,BATCH_SIZE,N_LAYERS,N_HEADS,D_MODEL,D_FF,val_acc_last,time_sec
0,local_global,0.0013,600,32,0.999,32,2,2,128,512,0.371094,24.847807
1,local_global,0.0010,600,32,0.999,32,2,2,128,512,0.347656,22.485089
2,local_global,0.0013,300,32,0.999,32,2,2,128,512,0.347656,21.937358
3,local_global,0.0010,200,32,0.999,32,2,2,128,512,0.347656,22.435025
4,local_global,0.0013,100,32,0.999,32,2,2,128,512,0.347656,22.706731
5,local_global,0.0010,300,32,0.999,32,2,2,128,512,0.343750,22.021667
6,local_global,0.0010,150,32,0.999,32,2,2,128,512,0.339844,19.624200
7,local_global,0.0013,150,32,0.999,32,2,2,128,512,0.339844,22.067634
8,local_global,0.0013,600,32,0.980,32,2,2,128,512,0.339844,23.231126
9,local_global,0.0013,200,32,0.980,32,2,2,128,512,0.339844,22.758952


Saved second-round results for all models to second_round_results_all_models.csv
